# P2-A8 — Evaluate Your RAG System (Phase 2 capstone)

You said it yourself in P2-A4: *"score retrieval and generation separately."* Now you do it. This bookends Phase 2 — the **eval skills from W2-A2** applied to the **RAG system from P2-A4**.

You'll measure three things on a labelled eval set:
1. **Retrieval recall** — did the right document make it into the top-k?
2. **Answer correctness** — is the final answer right? (using an *LLM-as-judge*, since exact string match is too brittle)
3. **Hallucination / abstention** — on questions NOT in the KB, did it correctly say *"I don't know"* instead of inventing an answer?

New technique: **LLM-as-judge** — using a model to grade outputs. It's how real teams eval open-ended LLM output at scale.

In [ ]:
# --- Setup (given): your P2-A4 RAG system, plus a labelled eval set ---
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
CHAT_MODEL = 'gpt-4o-mini'
EMBED_MODEL = 'text-embedding-3-small'

def embed(texts):
    if isinstance(texts, str):
        texts = [texts]
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data])

def ask(prompt):
    resp = client.chat.completions.create(model=CHAT_MODEL, messages=[{'role': 'user', 'content': prompt}])
    return resp.choices[0].message.content

kb = [
    "Zephyr Cloud's free tier includes 5 GB of storage and 100 GB of bandwidth per month.",   # 0
    "To upgrade your Zephyr plan, go to Billing > Plans and choose Pro or Enterprise.",        # 1
    "Zephyr Cloud's Pro plan costs $29 per month and includes 1 TB of storage.",              # 2
    "Zephyr support is available 24/7 via in-app chat for Pro and Enterprise customers.",      # 3
    "Zephyr's data centers are located in Oregon, Frankfurt, and Singapore.",                 # 4
]
kb_emb = embed(kb)

def retrieve(question, k=2):
    q = embed(question)[0]
    sims = (kb_emb @ q) / (np.linalg.norm(kb_emb, axis=1) * np.linalg.norm(q))
    top = np.argsort(sims)[::-1][:k]
    return [int(i) for i in top]              # return indices so we can check recall

def answer(question, k=2):
    ctx = "\n".join(kb[i] for i in retrieve(question, k))
    prompt = (f"Context:\n{ctx}\n\nQuestion: {question}\n\n"
              "Answer using ONLY the context above. If it's not there, say you don't know.")
    return ask(prompt)

# Labelled eval set: gold_doc = index of the doc that SHOULD be retrieved (None if out-of-KB)
eval_set = [
    {'question': 'How much does the Pro plan cost?',          'gold_doc': 2,    'expected': '$29 per month',                 'in_kb': True},
    {'question': 'How much storage does the free tier give?', 'gold_doc': 0,    'expected': '5 GB',                           'in_kb': True},
    {'question': 'Where are the data centers located?',       'gold_doc': 4,    'expected': 'Oregon, Frankfurt, Singapore',   'in_kb': True},
    {'question': 'How do I upgrade my plan?',                 'gold_doc': 1,    'expected': 'Billing > Plans',               'in_kb': True},
    {'question': 'Does Zephyr have a data center in Tokyo?',  'gold_doc': None, 'expected': "doesn't know / not in context", 'in_kb': False},
    {'question': 'Who is the CEO of Zephyr Cloud?',           'gold_doc': None, 'expected': "doesn't know / not in context", 'in_kb': False},
]
print('eval set:', len(eval_set), 'questions')

## Task 1 — Retrieval recall@k
For each **in-KB** question, retrieve the top-k indices and check whether its `gold_doc` is among them. Compute **recall@k** = fraction of in-KB questions whose gold doc was retrieved.
<br>Goal: this measures the *retrieval* half in isolation — if recall is low, no prompt fix will save you.

In [ ]:
# TODO: for each in-KB question, is gold_doc in retrieve(q)? compute recall@k


## Task 2 — Answer correctness via LLM-as-judge
Exact string match fails for open-ended answers ('$29/mo' vs 'The Pro plan is $29 per month'). So write a `judge(question, expected, actual)` that asks the LLM to grade: does `actual` correctly contain the `expected` fact? Return `True`/`False`.
<br>Run it on the **in-KB** questions and compute answer accuracy.
<br>Hint: prompt the judge to reply with a single word, e.g. *"Reply with exactly PASS or FAIL."*, then check the reply. (You're using the lessons from P2-A1: specificity + locked output format.)

In [ ]:
# TODO: def judge(question, expected, actual) -> bool ; score answer() on in-KB questions


## Task 3 — Hallucination check (the out-of-KB questions)
For the **out-of-KB** questions, the correct behaviour is to *abstain* ("I don't know"). Check each `answer()` and compute the **abstention rate** (how often it correctly refused) — equivalently, the hallucination rate is `1 - abstention`.
<br>Goal: a RAG system that confidently makes things up is worse than useless. This is the safety metric.

In [ ]:
# TODO: for out-of-KB questions, did answer() abstain? compute abstention/hallucination rate


## Task 4 — Assemble a report + explain
Build a pandas `DataFrame` (one row per eval question) with columns like: `question`, `in_kb`, `retrieved_ok`, `answer_correct` / `abstained`. Print it, plus your three headline metrics (recall@k, answer accuracy, hallucination rate).

Then, in words:
1. If a final answer is wrong, how do your separate metrics tell you whether retrieval or generation was at fault?
2. Name one concrete change you'd make to improve whichever number looks weakest — and how you'd re-measure to confirm it helped.

> _your answer here_